In [ ]:
!pip install tensorflow


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Download the dataset
!wget https://github.com/abisee/cnn-dailymail/archive/master.zip
!unzip master.zip
!mv cnn-dailymail-master cnn_dailymail

--2024-02-04 09:22:46--  https://github.com/abisee/cnn-dailymail/archive/master.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/abisee/cnn-dailymail/zip/refs/heads/master [following]
--2024-02-04 09:22:46--  https://codeload.github.com/abisee/cnn-dailymail/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 140.82.112.9
Connecting to codeload.github.com (codeload.github.com)|140.82.112.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘master.zip’

master.zip              [       <=>          ]  25.74M  7.16MB/s    in 3.6s    

2024-02-04 09:22:50 (7.16 MB/s) - ‘master.zip’ saved [26994149]

Archive:  master.zip
b15ad0a2db0d407a84b8ca9b5731e1f1c4bd24b9
   creating: cnn-dailymail-master/
  inflating: cnn-dailymail-master/LICENSE.md  
 

In [ ]:
#Load the data
def load_data(file_path):
  with open(file_path, 'r', encoding = 'utf-8') as file:
    data = file.readlines()
  return data

In [ ]:
train_article_path = '/content/drive/MyDrive/cnn-dailymail-master/url_lists/all_train.txt'
train_summary_path = '/content/drive/MyDrive/cnn-dailymail-master/url_lists/all_test.txt'

In [ ]:
train_article_data = load_data(train_article_path)
train_summary_data = load_data(train_summary_path)

In [ ]:
#Tokenization
max_input_length = 500
max_summary_length = 150
vocab_size = 5000

input_tokenizer = Tokenizer(num_words=vocab_size, oov_token = "<OOV>")
input_tokenizer.fit_on_texts(train_article_data)
input_sequences = input_tokenizer.texts_to_sequences(train_article_data)
input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
target_tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
target_tokenizer.fit_on_texts(train_summary_data)

target_sequences = target_tokenizer.texts_to_sequences(train_summary_data)
target_sequences = pad_sequences(target_sequences, maxlen=max_summary_length, padding = 'post')




In [1]:
# prompt: Do the summarization task on CNN daily mail dataset using BERT Model

!pip install transformers

import transformers
from transformers import BertTokenizer, TFBertForConditionalGeneration

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForConditionalGeneration.from_pretrained('bert-base-uncased')

def summarize_text(text):
  input_ids = tokenizer.encode(text, return_tensors='tf')
  output = model.generate(input_ids, max_length=150)
  summary = tokenizer.decode(output[0], skip_special_tokens=True)
  return summary

summarized_text = summarize_text(train_article_data[0])
print(summarized_text)


ImportError: cannot import name 'TFBertForConditionalGeneration' from 'transformers' (/usr/local/lib/python3.10/dist-packages/transformers/__init__.py)

In [ ]:
#Define the model
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=256, input_length=max_input_length, trainable=True))
model.add(LSTM(256, return_sequences=True))
model.add(Dense(vocab_size, activation='softmax'))

#compile the model
model.compile(optimizer = Adam(),loss=SparseCategoricalCrossentropy(),metrics = ['accuracy'])

In [ ]:
#Train the Model
model.fit(input_sequences, target_sequences, epochs = 5, batch_size=64, validation_split =0.2)

ValueError: Data cardinality is ambiguous:
  x sizes: 490048
  y sizes: 55072
Make sure all arrays contain the same number of samples.

In [ ]:
string = input("Enter a string: ")
number = int(input("Enter a number: "))
repeated_string = (string + '\n') * number
print("Result:\n" + repeated_string)

Enter a string: Manish
Enter a number: 8
Result:
Manish
Manish
Manish
Manish
Manish
Manish
Manish
Manish



In [ ]:
rows = 5
for i in range(0, rows):
    for j in range(0, i + 1):
        print("*", end=' ')
    print("\n")

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')

from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize


def rouge_1(system_summary, reference_summaries):
    # Tokenize system summary
    system_tokens = word_tokenize(system_summary.lower())
    # Tokenize reference summaries
    reference_tokens = [word_tokenize(ref.lower()) for ref in reference_summaries]

    # Calculate overlapping unigrams
    overlapping_unigrams = set(system_tokens) & set(reference_tokens[0])

    # Calculate precision
    precision = len(overlapping_unigrams) / len(system_tokens)
    # Calculate recall
    recall = len(overlapping_unigrams) / len(reference_tokens[0])

    # Calculate F1 score
    if precision + recall == 0:
        f1_score = 0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1_score

# Example usage
system_summary = "India's political landscape is criticized for lacking integrity and commitment, leading to a troubling situation. The author envisions an ideal leader as a combination of Medha Patkar, Anna Hazare, Narayana Murthy, and a part of V P Singh. Emphasizing the importance of commitment and integrity over cleverness, the author admires individuals with a do-or-die attitude. Despite acknowledging the leaders' imperfections, the focus is on their broad, positive vision. The author believes that figures like Patkar, Hazare, Singh, and Murthy, with their commitment and vision, could significantly benefit the country.."
reference_summaries = ["The complete lack of integrity and commitment on the part of all our political leaders has led to a dark and terrible situation in Indian politics. Political leaders today have a small, sectarian, straitjacketed vision and their own small little axes to grind. The consequences for the country, as we can see, are depressing. If I had to choose a political leader today, I would choose an amalgam of Medha Patkar, Anna Hazare, Narayana Murthy, and perhaps a small part of V P Singh. The main quality about someone like Medha Patkar is a do-or-die commitment, which would stand the country in good stead. We need people with integrity and commitment to be our leaders. Skills can be acquired along the way. Cleverness is a quality that helps people to help themselves, but doesn't do much good to the country at large. Finally it is commitment that really helps. One may disagree with Medha Patkar and Anna Hazare, but one cant question their commitment to their causes. Everyone makes mistakes, Jawaharlal Nehru made mistakes too. But if the overall vision is broad and positive, one can overlook these mistakes. V P Singh has a lot of failings. But I have always admired his vision. It covers a much broader spectrum than any other politician. Narayana Murthy has more or less the same qualities as Ms Patkar and Mr Hazare. His vision is large, and he has the tenacity to see his dreams and plans through. The contributions that I would expect from all these people would be very similar to the ones they are already making. They have all chalked out their paths in life and are ready to lay their lives on the line in reaching their targets. Such leadership would definitely do India a world of good"]

precision, recall, f1_score = rouge_1(system_summary, reference_summaries)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)


Precision: 0.3611111111111111
Recall: 0.11711711711711711
F1 Score: 0.17687074829931973


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# prompt: Write a code for checking the rouge score of the model, implement pegasus model for abstractive text

def rouge_score(model, input_sequences, target_sequences):
    # Generate summaries for each input sequence
    generated_summaries = model.predict(input_sequences)

    # Calculate ROUGE scores for each generated summary against its corresponding target summary
    rouge_scores = []
    for i in range(len(input_sequences)):
        precision, recall, f1_score = rouge_1(generated_summaries[i], [target_sequences[i]])
        rouge_scores.append(f1_score)

    # Calculate average ROUGE score
    average_rouge_score = sum(rouge_scores) / len(rouge_scores)

    return average_rouge_score

# Implement Pegasus model for abstractive text
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")

def pegasus_summarize(text):
    # Tokenize the input text
    input_ids = tokenizer.encode(text, return_tensors="pt")

    # Generate the summary
    output = model.generate(input_ids, max_length=50)

    # Decode the generated summary
    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    return summary

# Calculate ROUGE score for Pegasus model
average_rouge_score = rouge_score(pegasus_summarize, input_sequences, target_sequences)
print("Average ROUGE score:", average_rouge_score)


SyntaxError: invalid syntax (<ipython-input-9-4a8bf4b19221>, line 1)

In [ ]:
# prompt: Implement BERT model for abstractive text summarization

from transformers import BertTokenizer, BertForSequenceClassification

# Load the pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

def bert_summarize(text):
  # Tokenize the input text
  input_ids = tokenizer.encode(text, return_tensors='pt')

  # Generate the summary
  output = model(input_ids)[0]

  # Decode the generated summary
  summary = tokenizer.decode(output.argmax(dim=-1), skip_special_tokens=True)

  return summary

# Calculate ROUGE score for BERT model
average_rouge_score = rouge_score(bert_summarize, input_sequences, target_sequences)
print("Average ROUGE score:", average_rouge_score)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'rouge_score' is not defined

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize

def rouge_1(system_summary, reference_summaries):
    # Tokenization
    system_tokens = word_tokenize(system_summary.lower())
    # Tokenization of reference summaries
    reference_tokens = [word_tokenize(ref.lower()) for ref in reference_summaries]

    # Calculate overlapping unigrams
    overlapping_unigrams = set(system_tokens) & set(reference_tokens[0])

    # Calculate precision
    precision = len(overlapping_unigrams) / len(system_tokens)
    # Calculate recall
    recall = len(overlapping_unigrams) / len(reference_tokens[0])

    # Calculate F1 score
    if precision + recall == 0:
        f1_score = 0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1_score

# System summary
system_summary = "The Enforcement Directorate conducted raids in Ranchi, uncovering ₹25 crore unaccounted cash linked to Virendra Ram, former chief engineer. Video footage implicates Sanjiv Lal, personal secretary to Jharkhand minister Alamgir Alam. Alam refused to comment, while BJP calls for Election Commission action, suggesting election-related corruption. Raids spanned nine locations."
# Reference summaries
reference_summaries = "The Enforcement Directorate (ED) initiated a series of raids across various locations in Ranchi, the capital of Jharkhand, on Monday, resulting in the discovery of ₹25 crore in unaccounted cash. These recent raids, carried out under the Prevention of Money Laundering Act (PMLA), targeted around six locations associated with Virendra Ram, the former chief engineer at the Jharkhand Rural Development Department, and his close associates. Virendra Ram was apprehended by the ED in February 2023 in connection with a money laundering case. Video footage from the raid depicted a significant amount of currency notes strewn across a room allegedly belonging to the domestic helper of Sanjiv Lal, the personal secretary to Jharkhand Rural Development minister Alamgir Alam. Alamgir Alam, aged 70, is a Congress leader representing the Pakur constituency in the Jharkhand assembly. In response to the raids, Mr. Alam stated that it would be inappropriate to comment at this stage as the investigation by the probe agency is ongoing. Sanjiv Lal is a government employee and serves as my personal secretary. He has previously served as personal secretary to two former ministers. We typically appoint personal secretaries based on experience. It would not be appropriate to comment on the raids until the ED completes its investigation, he remarked, as quoted by news agency ANI. Corruption continues to persist in Jharkhand. The presence of such a large sum of money during elections suggests a plan to utilize these funds for electoral purposes. The Election Commission should take action on this matter, remarked Pratul Shahdev, spokesperson for the Jharkhand BJP. The probe agency is conducting simultaneous raids at nine locations, including Sail City in Ranchi. One ED team is currently searching Sail City to locate Vikas Kumar, an engineer from the Road Construction Department, while another team is conducting raids in the Bariatu, Morhabadi, and Bodia areas. precision, recall, f1_score = rouge_1(system_summary, reference_summaries)."
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)


Precision: 0.3611111111111111
Recall: 0.11711711711711711
F1 Score: 0.17687074829931973


In [ ]:
# prompt: Write code for NER with example

import spacy
from spacy import displacy

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

# Define the text to be analyzed
text = "I Love my Country, India and Narendra Modi is born in India."

# Perform named entity recognition
doc = nlp(text)

# Print the entities found
print("Entities:")
for entity in doc.ents:
    print(f"{entity.text}: {entity.label_}")

# Visualize the entities
displacy.render(doc, style="ent")



Entities:
India: GPE
Narendra Modi: PERSON
India: GPE


In [ ]:
import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")
text = "I Love my Country, India and Narendra Modi is born in India."
doc = nlp(text)
print("Entities:")
for entity in doc.ents:
    print(f"{entity.text}: {entity.label_}")
displacy.render(doc, style="ent")
print(doc.ents[0].text)
print(doc.ents[0].label_)
print(doc.ents[1].text)
print(doc.ents[1].label_)
print(doc.ents[2].text)
print(doc.ents[2].label_)


Entities:
India: GPE
Narendra Modi: PERSON
India: GPE


India
GPE
Narendra Modi
PERSON
India
GPE


In [ ]:
# prompt: Word2Vec program

import gensim
from gensim.models import Word2Vec

# Create a list of sentences
sentences = [['this', 'is', 'the', 'first', 'sentence'], ['this', 'is', 'the', 'second', 'sentence']]

# Create a Word2Vec model
model = Word2Vec(sentences, min_count=1)

# Print the vocabulary
print(model.wv.key_to_index)

# Print the vector for a word
print(model.wv['sentence'])

# Find the most similar words to a word
print(model.wv.most_similar('sentence'))

# Save the model
model.save('word2vec.model')

# Load the model
model = Word2Vec.load('word2vec.model')


{'sentence': 0, 'the': 1, 'is': 2, 'this': 3, 'second': 4, 'first': 5}
[-5.3622725e-04  2.3643136e-04  5.1033497e-03  9.0092728e-03
 -9.3029495e-03 -7.1168090e-03  6.4588725e-03  8.9729885e-03
 -5.0154282e-03 -3.7633716e-03  7.3805046e-03 -1.5334714e-03
 -4.5366134e-03  6.5540518e-03 -4.8601604e-03 -1.8160177e-03
  2.8765798e-03  9.9187379e-04 -8.2852151e-03 -9.4488179e-03
  7.3117660e-03  5.0702621e-03  6.7576934e-03  7.6286553e-04
  6.3508903e-03 -3.4053659e-03 -9.4640139e-04  5.7685734e-03
 -7.5216377e-03 -3.9361035e-03 -7.5115822e-03 -9.3004224e-04
  9.5381187e-03 -7.3191668e-03 -2.3337686e-03 -1.9377411e-03
  8.0774371e-03 -5.9308959e-03  4.5162440e-05 -4.7537340e-03
 -9.6035507e-03  5.0072931e-03 -8.7595852e-03 -4.3918253e-03
 -3.5099984e-05 -2.9618145e-04 -7.6612402e-03  9.6147433e-03
  4.9820580e-03  9.2331432e-03 -8.1579173e-03  4.4957981e-03
 -4.1370760e-03  8.2453608e-04  8.4986202e-03 -4.4621765e-03
  4.5175003e-03 -6.7869602e-03 -3.5484887e-03  9.3985079e-03
 -1.5776526e-0

In [ ]:
# prompt: use word2vec with simple example

# Create a list of sentences
sentences = [['this', 'is', 'the', 'first', 'sentence'], ['this', 'is', 'the', 'second', 'sentence']]

# Create a Word2Vec model
model = Word2Vec(sentences, min_count=1)

# Print the vocabulary
print(model.wv.key_to_index)

# Print the vector for a word
print(model.wv['sentence'])

# Find the most similar words to a word
print(model.wv.most_similar('sentence'))

# Save the model
model.save('word2vec.model')

# Load the model
model = Word2Vec.load('word2vec.model')


{'sentence': 0, 'the': 1, 'is': 2, 'this': 3, 'second': 4, 'first': 5}
[-5.3622725e-04  2.3643136e-04  5.1033497e-03  9.0092728e-03
 -9.3029495e-03 -7.1168090e-03  6.4588725e-03  8.9729885e-03
 -5.0154282e-03 -3.7633716e-03  7.3805046e-03 -1.5334714e-03
 -4.5366134e-03  6.5540518e-03 -4.8601604e-03 -1.8160177e-03
  2.8765798e-03  9.9187379e-04 -8.2852151e-03 -9.4488179e-03
  7.3117660e-03  5.0702621e-03  6.7576934e-03  7.6286553e-04
  6.3508903e-03 -3.4053659e-03 -9.4640139e-04  5.7685734e-03
 -7.5216377e-03 -3.9361035e-03 -7.5115822e-03 -9.3004224e-04
  9.5381187e-03 -7.3191668e-03 -2.3337686e-03 -1.9377411e-03
  8.0774371e-03 -5.9308959e-03  4.5162440e-05 -4.7537340e-03
 -9.6035507e-03  5.0072931e-03 -8.7595852e-03 -4.3918253e-03
 -3.5099984e-05 -2.9618145e-04 -7.6612402e-03  9.6147433e-03
  4.9820580e-03  9.2331432e-03 -8.1579173e-03  4.4957981e-03
 -4.1370760e-03  8.2453608e-04  8.4986202e-03 -4.4621765e-03
  4.5175003e-03 -6.7869602e-03 -3.5484887e-03  9.3985079e-03
 -1.5776526e-0

In [ ]:
# prompt: write a python program to tag part of speech in the sentence " I love my Country, My name is Roger, and want to build the application on chinese language

import nltk
from nltk import word_tokenize, pos_tag
import nltk
nltk.download('punkt')

# Sentence to be tagged
sentence = "I love my Country, My name is Roger, and want to build the application on chinese language"

# Tokenize the sentence
tokens = word_tokenize(sentence)

# Perform Part-of-Speech (POS) tagging
pos_tags = pos_tag(tokens)

# Print the results
for word, tag in pos_tags:
    print(f"{word}: {tag}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


LookupError: 
**********************************************************************
  Resource [93maveraged_perceptron_tagger[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('averaged_perceptron_tagger')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/averaged_perceptron_tagger/averaged_perceptron_tagger.pickle[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
import nltk
from nltk import word_tokenize, pos_tag

# Download necessary NLTK data
nltk.download('punkt')

# Sentence to be tagged
sentence = "I love my Country, My name is Roger, and want to build the application on Chinese language"

# Tokenize the sentence
tokens = word_tokenize(sentence)

# Perform Part-of-Speech (POS) tagging
pos_tags = pos_tag(tokens)

# Print the results
for word, tag in pos_tags:
    print(f"{word}: {tag}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LookupError: 
**********************************************************************
  Resource [93maveraged_perceptron_tagger[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('averaged_perceptron_tagger')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/averaged_perceptron_tagger/averaged_perceptron_tagger.pickle[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
import nltk
from nltk import word_tokenize, pos_tag

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Sentence to be tagged
sentence = "I love my Country, My name is Roger, and want to build the application on Chinese language"

# Tokenize the sentence
tokens = word_tokenize(sentence)

# Perform Part-of-Speech (POS) tagging
pos_tags = pos_tag(tokens)

# Print the results
for word, tag in pos_tags:
    print(f"{word}: {tag}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


I: PRP
love: VBP
my: PRP$
Country: NN
,: ,
My: NNP
name: NN
is: VBZ
Roger: NNP
,: ,
and: CC
want: VBP
to: TO
build: VB
the: DT
application: NN
on: IN
Chinese: JJ
language: NN


In [4]:
# prompt: Perform summarization on cnn daily mail dataset or any dataset

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from google.colab import drive
import transformers
from transformers import BertTokenizer, TFBertForConditionalGeneration
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
from transformers import BertTokenizer, BertForSequenceClassification
import spacy
from spacy import displacy
import gensim
from gensim.models import Word2Vec
from nltk import word_tokenize, pos_tag
!pip install tensorflow



drive.mount('/content/drive')
# Download the dataset
!wget https://github.com/abisee/cnn-dailymail/archive/master.zip
!unzip master.zip
!mv cnn-dailymail-master cnn_dailymail
#Load the data
def load_data(file_path):
  with open(file_path, 'r', encoding = 'utf-8') as file:
    data = file.readlines()
  return data
train_article_path = '/content/drive/MyDrive/cnn-dailymail-master/url_lists/all_train.txt'
train_summary_path = '/content/drive/MyDrive/cnn-dailymail-master/url_lists/all_test.txt'
train_article_data = load_data(train_article_path)
train_summary_data = load_data(train_summary_path)
#Tokenization
max_input_length = 500
max_summary_length = 150
vocab_size = 5000

input_tokenizer = Tokenizer(num_words=vocab_size, oov_token = "<OOV>")
input_tokenizer.fit_on_texts(train_article_data)
input_sequences = input_tokenizer.texts_to_sequences(train_article_data)
input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
target_tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
target_tokenizer.fit_on_texts(train_summary_data)

target_sequences = target_tokenizer.texts_to_sequences(train_summary_data)
target_sequences = pad_sequences(target_sequences, maxlen=max_summary_length, padding = 'post')




!pip install transformers


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForConditionalGeneration.from_pretrained('bert-base-uncased')

def summarize_text(text):
  input_ids = tokenizer.encode(text, return_tensors='tf')
  output = model.generate(input_ids, max_length=150)
  summary = tokenizer.decode(output[0], skip_special_tokens=True)
  return summary

summarized_text = summarize_text(train_article_data[0])
print(summarized_text)



#Define the model
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=256, input_length=max_input_length, trainable=True))
model.add(LSTM(256, return_sequences=True))
model.add(Dense(vocab_size, activation='softmax'))

#compile the model
model.compile(optimizer = Adam(),loss=SparseCategoricalCrossentropy(),metrics = ['accuracy'])
#Train the Model
model.fit(input_sequences, target_sequences, epochs = 5, batch_size=64, validation_split =0.2)
string = input("Enter a string: ")
number = int(input("Enter a number: "))
repeated_string = (string + '\n') * number
print("Result:\n" + repeated_string)
rows = 5
for i in range(0, rows):
    for j in range(0, i + 1):
        print("*", end=' ')
    print("\n")
!pip install nltk
nltk.download('punkt')



def rouge_1(system_summary, reference_summaries):
    # Tokenize system summary
    system_tokens = word_tokenize(system_summary.lower())
    # Tokenize reference summaries
    reference_tokens = [word_tokenize(ref.lower()) for ref in reference_summaries]

    # Calculate overlapping unigrams
    overlapping_unigrams = set(system_tokens) & set(reference_tokens[0])

    # Calculate precision
    precision = len(overlapping_unigrams) / len(system_tokens)
    # Calculate recall
    recall = len(overlapping_unigrams) / len(reference_tokens[0])

    # Calculate F1 score
    if precision + recall == 0:
        f1_score = 0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1_score

# Example usage
system_summary = "India's political landscape is criticized for lacking integrity and commitment, leading to a troubling situation. The author envisions an ideal leader as a combination of Medha Patkar, Anna Hazare, Narayana Murthy, and a part of V P Singh. Emphasizing the importance of commitment and integrity over cleverness, the author admires individuals with a do-or-die attitude. Despite acknowledging the leaders' imperfections, the focus is on their broad, positive vision. The author believes that figures like Patkar, Hazare, Singh, and Murthy, with their commitment and vision, could significantly benefit the country.."
reference_summaries = ["The complete lack of integrity and commitment on the part of all our political leaders has led to a dark and terrible situation in Indian politics. Political leaders today have a small, sectarian, straitjacketed vision and their own small little axes to grind. The consequences for the country, as we can see, are depressing. If I had to choose a political leader today, I would choose an amalgam of Medha Patkar, Anna Hazare, Narayana Murthy, and perhaps a small part of V P Singh. The main quality about someone like Medha Patkar is a do-or-die commitment, which would stand the country in good stead. We need people with integrity and commitment to be our leaders. Skills can be acquired along the way. Cleverness is a quality that helps people to help themselves, but doesn't do much good to the country at large. Finally it is commitment that really helps. One may disagree with Medha Patkar and Anna Hazare, but one cant question their commitment to their causes. Everyone makes mistakes, Jawaharlal Nehru made mistakes too. But if the overall vision is broad and positive, one can overlook these mistakes. V P Singh has a lot of failings. But I have always admired his vision. It covers a much broader spectrum than any other politician. Narayana Murthy has more or less the same qualities as Ms Patkar and Mr Hazare. His vision is large, and he has the tenacity to see his dreams and plans through. The contributions that I would expect from all these people would be very similar to the ones they are already making. They have all chalked out their paths in life and are ready to lay their lives on the line in reaching their targets. Such leadership would definitely do India a world of good"]

precision, recall, f1_score = rouge_1(system_summary, reference_summaries)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)


def rouge_score(model, input_sequences, target_sequences):
    # Generate summaries for each input sequence
    generated_summaries = model.predict(input_sequences)

    # Calculate ROUGE scores for each generated summary against its corresponding target summary
    rouge_scores = []
    for i in range(len(input_sequences)):
        precision, recall, f1_score = rouge_1(generated_summaries[i], [target_sequences[i]])
        rouge_scores.append(f1_score)

    # Calculate average ROUGE score
    average_rouge_score = sum(rouge_scores) / len(rouge_scores)

    return average_rouge_score

# Implement Pegasus model for abstractive text

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")

def pegasus_summarize(text):
    # Tokenize the input text
    input_ids = tokenizer.encode(text, return_tensors="pt")

    # Generate the summary
    output = model.generate(input_ids, max_length=50)

    # Decode the generated summary
    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    return summary

# Calculate ROUGE score for Pegasus model
rouge_score_pegasus = rouge_score(model, input_sequences, target_sequences)
print("ROUGE Score (Pegasus):", rouge_score_pegasus)


ImportError: cannot import name 'TFBertForConditionalGeneration' from 'transformers' (/usr/local/lib/python3.10/dist-packages/transformers/__init__.py)

In [ ]:
# prompt: Summarization with Transformer mentioned in Vaswani paper

import torch
from torch import nn

class TransformerSummarizer(nn.Module):
    def __init__(self, enc_inp_size, dec_inp_size, dec_out_size, N=6,
                   d_model=512, dim_feedforward=2048, num_heads=8, dropout=0.1):
        super(TransformerSummarizer, self).__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.N = N
        self.enc_inp_size = enc_inp_size
        self.dec_inp_size = dec_inp_size
        self.dec_out_size = dec_out_size

        self.encoder = nn.ModuleList([deepcopy(
            EncoderLayer(d_model, num_heads, dim_feedforward, dropout))
            for _ in range(N)])
        self.decoder = nn.ModuleList([deepcopy(
            DecoderLayer(d_model, num_heads, dim_feedforward, dropout))
            for _ in range(N)])
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.pos_dec = PositionalEncoding(d_model, dropout)
        self.src_embed = nn.Linear(enc_inp_size, d_model)
        self.tgt_embed = nn.Linear(dec_inp_size, d_model)
        self.out = nn.Linear(d_model, dec_out_size)

        self.init_weights()

    def forward(self, src, trg, src_mask, trg_mask):
        src_emb = self.pos_enc(self.src_embed(src))
        trg_emb = self.pos_dec(self.tgt_embed(trg))

        for i in range(self.N):
            src_emb = self.encoder[i](src_emb, src_mask)

        for i in range(self.N):
            trg_emb = self.decoder[i](trg_emb, src_emb, trg_mask)

        output = self.out(trg_emb)
        return output

    def init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)


In [5]:
# prompt: Summarization using seq2seq model on cnn daily mail dataset

# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the dataset
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/cnn_dailymail/dailymail/dailymail.csv')

# Preprocess the data
df['article'] = df['article'].apply(lambda x: ' '.join(x.split(' ')[:500]))
df['highlights'] = df['highlights'].apply(lambda x: ' '.join(x.split(' ')[:100]))

# Tokenize the data
tokenizer_article = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer_article.fit_on_texts(df['article'])
article_sequences = tokenizer_article.texts_to_sequences(df['article'])
article_padded = pad_sequences(article_sequences, maxlen=500)

tokenizer_highlights = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer_highlights.fit_on_texts(df['highlights'])
highlights_sequences = tokenizer_highlights.texts_to_sequences(df['highlights'])
highlights_padded = pad_sequences(highlights_sequences, maxlen=100)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(article_padded, highlights_padded, test_size=0.2, random_state=42)

# Define the model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=256, input_length=500))
model.add(LSTM(256, return_sequences=True))
model.add(LSTM(256))
model.add(Dense(100, activation='relu'))
model.add(Dense(5000, activation='softmax'))

# Compile the model
model.compile(optimizer=Adam(), loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

# Evaluate the model
model.evaluate(X_test, y_test)

# Generate summaries for new articles
new_articles = ['India is a great country with a rich history and culture.', 'The Taj Mahal is one of the most beautiful buildings in the world.']
new_article_sequences = tokenizer_article.texts_to_sequences(new_articles)
new_article_padded = pad_sequences(new_article_sequences, maxlen=500)
summaries = model.predict(new_article_padded)
summaries_decoded = tokenizer_highlights.sequences_to_texts(summaries)

# Print the summaries
for i in range(len(new_articles)):
    print('Article:', new_articles[i])
    print('Summary:', summaries_decoded[i])


Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cnn_dailymail/dailymail/dailymail.csv'